In [ ]:
import os, sys
# Add project root to sys.path so we can import models and pipelines
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)


# PRISM: CNN Cough Detector Training
This notebook trains the PRISM CNN on a free Google Colab GPU.

**Prerequisites:**
Make sure you have uploaded the following to the root of your Google Drive:
1. `prism-colab` (folder containing the code)
2. `datasets-features.zip` (the 6GB zip file with the features)

In [ ]:
# 1. Setup Environment and Mount Drive
!pip install -q loguru rich scikit-learn pyyaml

import os

from google.colab import drive

# Mount Google Drive (this will ask for permission)
drive.mount('/content/drive')

# Symlink the PRISM code into the Colab working directory
if not os.path.exists('/content/prism'):
    os.symlink('/content/drive/MyDrive/prism-colab', '/content/prism')
if not os.path.exists('/content/models'):
    os.symlink('/content/drive/MyDrive/prism-colab/models', '/content/models')

print("\n✅ Code setup complete!")

In [ ]:
# 2. Unzip Features (Run this ONLY ONCE per session)
# This takes about 2-3 minutes to extract 131,000 files to the fast local Colab disk
import os

if not os.path.exists('/content/features/manifest.csv'):
    print("⏳ Unzipping features to local SSD... This will take a few minutes.")
    !unzip -q /content/drive/MyDrive/datasets-features.zip -d /content/features
    print("✅ Features unzipped!")
else:
    print("✅ Features already unzipped.")

In [ ]:
# 3. Verify GPU Acceleration
import torch

if torch.cuda.is_available():
    print(f"🚀 GPU Active: {torch.cuda.get_device_name(0)}")
else:
    print("❌ ERROR: No GPU found. Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

In [ ]:
# 4. Train the Model!
import os

os.chdir('/content/prism')

!python -m models.cough_detector.run_training \
    --manifest /content/features/manifest.csv \
    --features-dir /content/features \
    --epochs 50 \
    --batch-size 128

In [ ]:
# 5. Save the Checkpoint Back to Google Drive
!cp checkpoints/cough_detector_best.pt /content/drive/MyDrive/prism-colab/checkpoints/
print("✅ Checkpoint safely copied to Google Drive!")
print("You can now download it to your local machine.")